Notes:
- YOLO26 is available in 5 sizes: Nano (n), Small (s), Medium (m), Large (l), and Extra Large (x). 
- YOLO26 integrates ProgLoss and Small Target-Aware Label Assignment (STAL) which improve small-object detection accuracy [Arxiv link](https://arxiv.org/pdf/2509.25164)
- Re data prep: YOLO object detection will require:
    - images
    - labels (one .txt file per image in YOLO format): 
        - <class_id> <x_center> <y_center> <width> <height>
        - Roboflow supports YOLO26 for labeling, training, and deployment, so it would be a good annotation tool
            - if we want a free, open source alternative, we can look into Label Studio
- We don't need to create a dataset configuration YAML because roboflow already gives us pre split train/val/test folders, correctly formatted .txt labels, and a ready made data.yaml with class names, paths, nc already filled in

Hardware agnostic setup (may not need/may need to adjust code to take advantage of this)

In [ ]:
import os
import torch

# Hardware-Agnostic Device Selection
if torch.cuda.is_available():
    device = torch.device("cuda") # Lenovo / WSL2 (NVIDIA GPU)
    
    # Dynamic Worker Allocation Formula
    # Reserve 2 cores for Windows/WSL; cap at 12 to avoid over-allocation (pytorch performance tends to plateau beyond 8 workers)
    total_cores = os.cpu_count()
    NUM_WORKERS = min(12, max(1, os.cpu_count() - 2))
    IS_CUDA = True

elif torch.backends.mps.is_available():
    device = torch.device("mps") # Mac Apple Silicon
    NUM_WORKERS = 0 # Multiprocessing on Mac MPS can hang; set to 0 for safety
    IS_CUDA = False

else:
    device = torch.device("cpu") # Fallback to CPU
    NUM_WORKERS = 0 # Multiprocessing on CPU can be less efficient; set to 0 for simplicity
    IS_CUDA = False

print(f"Using device: {device}")
print(f"Using {NUM_WORKERS} worker(s) for data loading")
print(f"CUDA Available for Mixed Precision: {IS_CUDA}")

### Fine-Tuning the Model with Transfer Learning

[Python Usage Documentation](https://docs.ultralytics.com/usage/python/#train)

I am testing all of this here/ drafting here but I will eventually put it into a training script that picks up after download_data.py

In [ ]:
from roboflow import Roboflow

rf = Roboflow(api_key=os.environ["ROBOFLOW_API_KEY"])

In [ ]:
from ultralytics import YOLO

# load a pretrained YOLO large model (WE COULD START SMALL OR MEDIUM AND MOVE UP IF VALIDATION LOSS PLATEAUS and we have enough data)
model = YOLO('yolo26l.pt')

# fine tune on shark dataset
results = model.train(
    data = "data.yaml", # WILL UPDATE PATH ONCE I HAVE DATA
    epochs = @TODO,
    imgsz = @TODO,
    batch = @TODO,
    lr0 = @TODO, # want a lower LR for fine tuning
    freeze = @TODO, # freeze first ? (backbone) layers initially
    patience = @TODO, # early stopping parameter
    augment = True # our dataset may be large enough to not need this
)


Draft for script:

In [ ]:
# TODO: COMPLETE THIS SCRIPT AND MAKE UPDATES NEEDED

import os
from pathlib import Path
from ultralytics import YOLO

# --- configure variables ---
# TODO: CHANGE / DECIDE ON THESE PARAMETERS (everything is currently a placeholder)
DATA_ROOT = Path("../DS6050_ML3_Final_Proj/data/raw")
YAML_PATH = DATA_ROOT / "data.yaml"
MODEL_SIZE = "l"        # n, s, m, l, x
EPOCHS = 100
IMG_SIZE = 640          
BATCH_SIZE = 16
FREEZE = 10             # TODO: Decide if we are freezing the backbone or fine tuning the whole network; freeze backbone layers for transfer learning
RUN_NAME = "shark_v1"   # name of the run

# verify data exists
if not YAML_PATH.exists():
    raise FileNotFoundError(
        f"data.yaml not found at {YAML_PATH}. Run `python scripts/download_data.py` first."
    )

# --- initialize model ---
# load a pretrained YOLO large model
model = YOLO(f"yolo26{MODEL_SIZE}.pt")

# --- training loop ---
results = model.train(
    data = str(YAML_PATH),
    epochs = EPOCHS,
    imgsz = IMG_SIZE,
    batch = BATCH_SIZE,
    lr0 = 0.001,            # want a lower LR for fine tuning
    freeze = FREEZE,        
    patience = 10,          # early stopping if validation loss plateaus
    augment = True,         # since we did not augment during roboflow export, may want to do
    name = RUN_NAME,
    exist_ok = True,        # overwrite existing runs (won't crash if run name already exists)
)

print(f"Best weights saved to: {model.trainer.best}")